# Hadoop HDFS Commands Lab

This notebook runs on the **BigDataCluster** (Hadoop 3.4.2, HDFS in high availability, 3 DataNodes).

**How to open it**
1. Start the cluster: `docker compose up -d` in the `BigDataCluster` folder.
2. Open Jupyter at <http://localhost:8000>. Jupyter runs on the `worker4` container.
3. Open this notebook. Run each cell with **Shift + Enter**, from top to bottom.

**How the cells work**
* A line that starts with `!` runs as a Linux shell command on `worker4`.
* To change folder for the rest of the notebook, use Python's `os.chdir(...)`. `!cd` would not work, because each `!` line runs in its own shell.
* You can also type the same commands in a terminal without the `!`. Open one with `docker exec -it worker1 bash` from PowerShell.

> **Two file systems.** `ls`, `cat`, `mkdir` (no `hdfs dfs`) work on the **local Linux disk** of the container.
> `hdfs dfs -ls`, `hdfs dfs -cat` and the rest work on **HDFS**, the distributed file system shared by the whole cluster.

## 0. Set up a local working folder

We keep the local files for this lab in `~/hdfs_lab` on worker4.

In [ ]:
import os
!mkdir -p ~/hdfs_lab
os.chdir(os.path.expanduser("~/hdfs_lab"))
!pwd
!whoami
!hdfs version | head -n 1

You are logged in as `user`. On this cluster `user` started the NameNode, so it is the **HDFS superuser**:
it can create folders anywhere, including directly under `/`.
That is why this lab does not need `su` or `su - hdfs` (those were for the Cloudera VM).

In [ ]:
!hdfs dfs -ls -d /

## 1. Get help for a command

`-help` describes one command in detail. `-usage` shows only its syntax.
Running `hdfs dfs -help` with no command name prints the help for every command, which is very long.

In [ ]:
!hdfs dfs -help ls

In [ ]:
!hdfs dfs -usage mkdir

## 2. List a directory: `-ls`

Syntax: `hdfs dfs -ls <path>`

In [ ]:
!hdfs dfs -ls /

In [ ]:
!hdfs dfs -ls /user

Reading a line of `-ls` output:

| Column | Example | Meaning |
|---|---|---|
| 1 | `drwxr-xr-x` | `d` = directory, then permissions for owner, group, others |
| 2 | `-` or `1` | Replication factor (folders show `-`) |
| 3, 4 | `user supergroup` | Owner and group |
| 5 | `41` | Size in bytes |
| 6, 7 | `2026-10-16 01:04` | Last modified |
| 8 | `/lab1dir/lab1.txt` | Path |

## 3. Create a directory: `-mkdir`

Syntax: `hdfs dfs -mkdir <path>`

`-p` creates any missing parent folders and does not complain if the folder already exists, so the cell is safe to run twice.

In [ ]:
!hdfs dfs -mkdir -p /lab1dir
!hdfs dfs -ls /

## 4. Create local files to work with

`%%writefile` saves the cell's text into a file on the local disk of worker4.

In [ ]:
%%writefile lab1.txt
hello world
hello hadoop
hello hdfs

In [ ]:
%%writefile lab123.txt
this is lab123
it will be copied and moved around in HDFS

In [ ]:
!ls -l
!cat lab1.txt

## 5. Upload a local file to HDFS: `-put` and `-copyFromLocal`

Syntax: `hdfs dfs -put <local source> <HDFS destination>`

`-f` overwrites the file if it is already in HDFS. Without `-f` a second upload fails with `File exists`.

In [ ]:
!hdfs dfs -put -f lab1.txt /lab1dir
!hdfs dfs -ls /lab1dir

`-copyFromLocal` does the same as `-put`, but the source must be a local file.

Syntax: `hdfs dfs -copyFromLocal <local source> <HDFS destination>`

In [ ]:
!hdfs dfs -copyFromLocal -f lab123.txt /lab1dir
!hdfs dfs -ls /lab1dir

There is also `-moveFromLocal`. It uploads the file and then deletes the local copy.

## 6. Show the content of a file: `-cat`

Syntax: `hdfs dfs -cat <HDFS path>`

In [ ]:
!hdfs dfs -cat /lab1dir/lab1.txt

## 7. Download from HDFS to the local disk: `-get` and `-copyToLocal`

Syntax: `hdfs dfs -get <HDFS source> <local destination>`

Put quotes around a path with `*` in it. The quotes stop the local shell from expanding the `*`, so HDFS matches the files itself.

In [ ]:
!mkdir -p downloads
!hdfs dfs -get -f "/lab1dir/*.txt" downloads/
!ls -l downloads

`-copyToLocal` does the same as `-get`.

In [ ]:
!hdfs dfs -copyToLocal -f /lab1dir/lab1.txt downloads/lab1-copy.txt
!cat downloads/lab1-copy.txt

## 8. Copy inside HDFS: `-cp`

Syntax: `hdfs dfs -cp <HDFS source> <HDFS destination>`

The file stays in its original place and a second copy is created.

In [ ]:
!hdfs dfs -cp -f /lab1dir/lab123.txt /tmp
!hdfs dfs -ls /tmp

## 9. Move or rename inside HDFS: `-mv`

Syntax: `hdfs dfs -mv <HDFS source> <HDFS destination>`

After the move, the file is no longer in `/tmp`.
If you run this cell a second time it fails with `File exists`, because the file is already in `/user`. The clean-up in section 16 resets everything.

In [ ]:
!hdfs dfs -mv /tmp/lab123.txt /user
!hdfs dfs -ls /user

`-mv` also renames a file when source and destination are in the same folder:

In [ ]:
!hdfs dfs -mv /user/lab123.txt /user/lab123-renamed.txt
!hdfs dfs -ls /user

## 10. Add data to the end of a file: `-appendToFile`

Syntax: `hdfs dfs -appendToFile <local source> <HDFS file>`

HDFS files are **write once, read many**: you cannot change the middle of a file, but you can append to its end.

In [ ]:
%%writefile appendtxt.txt
this line was appended

In [ ]:
!cat appendtxt.txt
!hdfs dfs -appendToFile appendtxt.txt /lab1dir/lab1.txt
!hdfs dfs -cat /lab1dir/lab1.txt

Use `-` as the source to append whatever another command prints:

In [ ]:
!echo "appended from echo" | hdfs dfs -appendToFile - /lab1dir/lab1.txt
!hdfs dfs -cat /lab1dir/lab1.txt

## 11. Check space: `-df` and `-du`

`-df` shows the capacity of the whole file system. `-du` shows how much space a path uses.
`-h` prints sizes as KB, MB, GB instead of bytes.

In [ ]:
!hdfs dfs -df -h /

In [ ]:
!hdfs dfs -du -h /lab1dir

`hdfs://bdss` is the name of the whole high-availability cluster, not of one machine.

## 12. Change permissions: `-chmod`

Syntax: `hdfs dfs -chmod <mode> <HDFS path>`

In [ ]:
!hdfs dfs -ls /lab1dir
!hdfs dfs -chmod 777 /lab1dir/lab1.txt
!hdfs dfs -ls /lab1dir

### Understanding permissions

`777` is a *UNIX-style* permission code. It has three digits, one each for **owner**, **group** and **others**:

```
hdfs dfs -chmod   7      7      7
                  ↓      ↓      ↓
                Owner  Group  Others
```

Each digit is the sum of:

| Number | Permission | Meaning |
|---|---|---|
| 4 | Read (r) | Can view the file contents |
| 2 | Write (w) | Can modify or delete the file |
| 1 | Execute (x) | Can run the file (script or program), or enter a directory |

So:
* `7` = 4+2+1 = **read + write + execute** → `rwx`
* `6` = 4+2 = **read + write** → `rw-`
* `5` = 4+1 = **read + execute** → `r-x`
* `4` = **read only** → `r--`
* `0` = **no permission** → `---`

**Example:** `hdfs dfs -chmod 777 /lab1dir/lab1.txt` gives everyone read, write and execute. `-ls` then shows `-rwxrwxrwx`.

| User type | Permissions |
|---|---|
| Owner | read + write + execute |
| Group | read + write + execute |
| Others | read + write + execute |

### Common modes

| Mode | Shows as | Who can do what | Typical use |
|---|---|---|---|
| 777 | `rwxrwxrwx` | Everyone can do everything | Very open, avoid |
| 755 | `rwxr-xr-x` | Owner full, others read and enter | Folders, scripts |
| 644 | `rw-r--r--` | Owner read/write, others read | Shared text files |
| 600 | `rw-------` | Only the owner | Private files |

Try two more modes and compare the first column:

In [ ]:
!hdfs dfs -chmod 644 /lab1dir/lab1.txt
!hdfs dfs -chmod 600 /lab1dir/lab123.txt
!hdfs dfs -ls /lab1dir

`-R` changes a folder and everything inside it:

In [ ]:
!hdfs dfs -chmod -R 755 /lab1dir
!hdfs dfs -ls /lab1dir

## 13. More useful commands

| Command | What it does |
|---|---|
| `-head` / `-tail` | First / last kilobyte of a file |
| `-count` | Number of folders, files and bytes under a path |
| `-stat` | Chosen details of a file (`%r` replication, `%o` block size, `%b` bytes) |
| `-test -e` | Exit code 0 if the path exists; useful in scripts |
| `-touchz` | Create an empty file |

In [ ]:
!hdfs dfs -head /lab1dir/lab1.txt

In [ ]:
!hdfs dfs -count /lab1dir
!hdfs dfs -stat "replication=%r blocksize=%o bytes=%b name=%n" /lab1dir/lab1.txt

In [ ]:
!hdfs dfs -touchz /lab1dir/empty.txt
!hdfs dfs -test -e /lab1dir/empty.txt && echo "empty.txt exists"
!hdfs dfs -ls /lab1dir

## 14. Delete a file: `-rm`

Syntax: `hdfs dfs -rm <HDFS path>`

> ⚠️ **No trash on this cluster.** The trash is turned off (`fs.trash.interval = 0`, see the next cell).
> `-rm` deletes **immediately and permanently**. Check the path with `-ls` before you delete.

In [ ]:
!hdfs getconf -confKey fs.trash.interval

In [ ]:
!hdfs dfs -rm /user/lab123-renamed.txt
!hdfs dfs -ls /user

## 15. Delete a folder: `-rm -r` and `-rmdir`

| Command | Use it for |
|---|---|
| `hdfs dfs -rm -r <folder>` | Delete a folder **and everything inside it** |
| `hdfs dfs -rm -r -f <folder>` | Same, with no error if the folder does not exist |
| `hdfs dfs -rmdir <folder>` | Delete a folder only if it is **empty** (the safe option) |

Without `-r`, `-rm` refuses to delete a folder.

In [ ]:
!hdfs dfs -mkdir -p /lab1dir/empty_folder /lab1dir/full_folder
!hdfs dfs -cp /lab1dir/lab1.txt /lab1dir/full_folder/
!hdfs dfs -ls /lab1dir

In [ ]:
!hdfs dfs -rmdir /lab1dir/empty_folder
!hdfs dfs -rm -r /lab1dir/full_folder
!hdfs dfs -ls /lab1dir

## 16. Clean up

This removes everything the lab created in HDFS and on the local disk, so you can run the notebook again from the top.

In [ ]:
!hdfs dfs -rm -r -f /lab1dir /tmp/lab123.txt /user/lab123.txt /user/lab123-renamed.txt
!hdfs dfs -ls /
os.chdir(os.path.expanduser("~"))
!rm -rf ~/hdfs_lab

## Cheat sheet

| Task | Command |
|---|---|
| Help for one command | `hdfs dfs -help ls` |
| List | `hdfs dfs -ls /path` |
| Make folder | `hdfs dfs -mkdir -p /path` |
| Upload | `hdfs dfs -put local.txt /path` |
| Download | `hdfs dfs -get /path/file.txt localdir/` |
| Show content | `hdfs dfs -cat /path/file.txt` |
| Copy in HDFS | `hdfs dfs -cp /a /b` |
| Move / rename in HDFS | `hdfs dfs -mv /a /b` |
| Append | `hdfs dfs -appendToFile local.txt /path/file.txt` |
| Space | `hdfs dfs -df -h /` · `hdfs dfs -du -h /path` |
| Permissions | `hdfs dfs -chmod 644 /path/file.txt` |
| Delete file (permanent here) | `hdfs dfs -rm /path/file.txt` |
| Delete folder (permanent here) | `hdfs dfs -rm -r /path/folder` |
| Delete empty folder | `hdfs dfs -rmdir /path/folder` |

**Common mistakes**
* `–ls` with a long dash (–) copied from Word or slides fails. Type a normal hyphen `-`.
* Mixing up local and HDFS paths: `ls` shows the local disk, `hdfs dfs -ls` shows HDFS.
* `File exists`: add `-f` to `-put`, `-get` or `-cp` to overwrite.